# Story Embedding Pipeline

This notebook records the full process of converting raw story transcripts into
feature matrices for fMRI ridge regression.

**Three-stage pipeline:** Embed → Downsample → Trim & Lag

**Final output shape:** `(T', embed_dim × 4)` where T' = trimmed fMRI timepoints

---

## 0. Setup & Imports

In [31]:
import sys, os
import numpy as np
import pickle

# Paths
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('./provided'))

from provided.preprocessing import downsample_word_vectors, make_delayed
from preprocessing_utils import preprocess_embeddings, load_fmri

# Directories
DATA_PATH  = '/scratch/users/s214/lab3'   # adjust to your data root
RAW_DIR    = '../data/raw'
EMBED_DIR  = '../data/embeddings'
os.makedirs(EMBED_DIR, exist_ok=True)

# Subjects and train/test split
SUBJECTS      = ['subject2', 'subject3']
SUBJECT_IDS   = {'subject2': 's2', 'subject3': 's3'}

with open('../data/train_test_split.json') as f:
    split = __import__('json').load(f)
TRAIN_STORIES = split['train']
TEST_STORIES  = split['test']
ALL_STORIES   = TRAIN_STORIES + TEST_STORIES

print(f'Train stories ({len(TRAIN_STORIES)}): {TRAIN_STORIES}')
print(f'Test  stories ({len(TEST_STORIES)}):  {TEST_STORIES}')

Train stories (81): ['mybackseatviewofagreatromance', 'becomingindian', 'cocoonoflove', 'goldiethegoldfish', 'theclosetthatateeverything', 'quietfire', 'lifeanddeathontheoregontrail', 'thepostmanalwayscalls', 'findingmyownrescuer', 'sloth', 'legacy', 'treasureisland', 'learninghumanityfromdogs', 'naked', 'hangtime', 'lawsthatchokecreativity', 'jugglingandjesus', 'whyimustspeakoutaboutclimatechange', 'thetiniestbouquet', 'thumbsup', 'breakingupintheageofgoogle', 'gpsformylostidentity', 'escapingfromadirediagnosis', 'howtodraw', 'ifthishaircouldtalk', 'catfishingstrangerstofindmyself', 'seedpotatoesofleningrad', 'odetostepfather', 'notontheusualtour', 'backsideofthestorm', 'golfclubbing', 'thefreedomridersandme', 'kiksuya', 'afatherscover', 'life', 'canplanetearthfeedtenbillionpeoplepart2', 'haveyoumethimyet', 'myfathershands', 'swimmingwithastronauts', 'comingofageondeathrow', 'avatar', 'exorcism', 'alternateithicatom', 'tildeath', 'shoppinginchina', 'theinterview', 'cautioneating', 'st

## 1. Pipeline Parameters

| Parameter | Value | Reason |
|---|---|---|
| `TR` | 2 s | fMRI repetition time — one scan per 2 s |
| `TRIM_START` | 5 s = 2 TRs | Scanner warmup / subject settling |
| `TRIM_END` | 10 s = 5 TRs | BOLD lags audio; last words not fully captured |
| `DELAYS` | [1, 2, 3, 4] TRs | Hemodynamic response peaks ~4–6 s after stimulus |

In [32]:
from preprocessing_utils import TR, TRIM_START, TRIM_END, DELAYS

print(f'TR          = {TR} s')
print(f'TRIM_START  = {TRIM_START} s  ({int(TRIM_START/TR)} TRs)')
print(f'TRIM_END    = {TRIM_END} s  ({int(TRIM_END/TR)} TRs)')
print(f'DELAYS      = {DELAYS} TRs')

TR          = 2 s
TRIM_START  = 5 s  (2 TRs)
TRIM_END    = 10 s  (5 TRs)
DELAYS      = [1, 2, 3, 4] TRs


## 2. Load Word Sequences (WordSeqs)

`wordseqs` is a dict `{story: DataSequence}`.  
Each `DataSequence` contains:
- `.data` — list of word tokens in the story
- `.data_times` — timestamp of each word (seconds)
- `.tr_times` — fMRI scan timestamps (seconds)

These are used in Stage 2 (Lanczos downsampling) to align words to TR timepoints.

In [33]:
wordseqs = pickle.load(open(os.path.join(DATA_PATH, 'raw_text.pkl'), 'rb'))

# Inspect one story
example_story = ALL_STORIES[0]
ds = wordseqs[example_story]
print(f'Story: {example_story}')
print(f'  Words: {len(ds.data)}')
print(f'  First 10 words: {ds.data[:10]}')
print(f'  Word timestamps (first 5): {ds.data_times[:5]}')
print(f'  TR timestamps  (first 5): {ds.tr_times[:5]}')

Story: mybackseatviewofagreatromance
  Words: 1794
  First 10 words: ['', 'my', 'plan', 'was', 'to', 'go', 'to', 'morocco', 'to', 'escape']
  Word timestamps (first 5): [0.00623583 1.50408163 1.78843537 2.03786848 2.14761905]
  TR timestamps  (first 5): [-9. -7. -5. -3. -1.]


/tmp/ipykernel_3904596/2078775971.py:1: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  wordseqs = pickle.load(open(os.path.join(DATA_PATH, 'raw_text.pkl'), 'rb'))


### fMRI Reaction Data Dimensions for Example Story

Each fMRI file has shape `(T, V)` where:
- `T` = number of TR timepoints (varies by story length)
- `V` = number of voxels = 94251 (whole brain, fixed)

The **fMRI is the fixed target** — we never modify it. Instead, we transform the story embedding to match:
1. **Crop** the downsampled embedding to `T_fmri` (wordseq may have slightly more TR entries than the actual fMRI scan)
2. **Trim** the story embedding by removing the first 2 and last 5 TRs
3. **Delay** to produce the final feature matrix aligned to fMRI timepoints

In [ ]:
n_words   = len(wordseqs[example_story].data)
n_trs_seq = len(wordseqs[example_story].tr_times)

print(f"Example story: {example_story}")
print(f"  words          : {n_words}")
print(f"  TR timepoints  : {n_trs_seq}")

# Subject fMRI — raw dimensions only, no trimming applied here
for subject in SUBJECTS:
    fmri = np.load(os.path.join(DATA_PATH, subject, f"{example_story}.npy"), mmap_mode="r")
    print(f"[{subject}]  fMRI raw shape: {fmri.shape}  (T={fmri.shape[0]}, voxels={fmri.shape[1]})")

Example story: mybackseatviewofagreatromance
  words          : 1794
  TR timepoints  : 399

[subject2]  fMRI raw shape: (384, 94251)  (T=384, voxels=94251)
[subject3]  fMRI raw shape: (384, 95556)  (T=384, voxels=95556)


---
## Stage 1 — Word-level Embedding

Replace each word token with a dense vector from a pre-trained model.  
OOV (out-of-vocabulary) words receive a **zero vector**.

Output: `{story: np.ndarray of shape (n_words, embed_dim)}`

### 1a. Load GloVe Vectors

Pre-trained model: **GloVe 840B Common Crawl, 300-d**  
File: `data/raw/glove.840B.300d.txt`  
Format: one line per word — `word f1 f2 ... f300`

In [37]:
GLOVE_PATH = os.path.join(RAW_DIR, 'glove.840B.300d.txt')
GLOVE_DIM  = 300

print(f'Loading GloVe from {GLOVE_PATH} ...')
glove = {}
with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.rstrip().split(' ')
        glove[parts[0]] = np.array(parts[1:], dtype=np.float32)

print(f'Loaded {len(glove):,} GloVe vectors (dim={GLOVE_DIM})')

Loading GloVe from ../data/raw/glove.840B.300d.txt ...
Loaded 2,196,016 GloVe vectors (dim=300)


### 1b. Embed Each Word Token

In [40]:
word_vectors = {}
oov_total = 0

for story in ALL_STORIES:
    vecs = []
    for word in wordseqs[story].data:
        w = word.lower()
        if w in glove:
            vecs.append(glove[w])
        else:
            vecs.append(np.zeros(GLOVE_DIM, dtype=np.float32))
            oov_total += 1
    word_vectors[story] = np.array(vecs, dtype=np.float32)

print(f'OOV tokens: {oov_total}')
for story in ALL_STORIES:
    print(f'  {story}: word matrix shape = {word_vectors[story].shape}')

OOV tokens: 4092
  mybackseatviewofagreatromance: word matrix shape = (1794, 300)
  becomingindian: word matrix shape = (2619, 300)
  cocoonoflove: word matrix shape = (1984, 300)
  goldiethegoldfish: word matrix shape = (1680, 300)
  theclosetthatateeverything: word matrix shape = (1928, 300)
  quietfire: word matrix shape = (1905, 300)
  lifeanddeathontheoregontrail: word matrix shape = (2389, 300)
  thepostmanalwayscalls: word matrix shape = (2220, 300)
  findingmyownrescuer: word matrix shape = (1498, 300)
  sloth: word matrix shape = (2403, 300)
  legacy: word matrix shape = (1893, 300)
  treasureisland: word matrix shape = (1763, 300)
  learninghumanityfromdogs: word matrix shape = (1484, 300)
  naked: word matrix shape = (3218, 300)
  hangtime: word matrix shape = (1927, 300)
  lawsthatchokecreativity: word matrix shape = (2084, 300)
  jugglingandjesus: word matrix shape = (887, 300)
  whyimustspeakoutaboutclimatechange: word matrix shape = (2336, 300)
  thetiniestbouquet: word 

---
## Stage 2 — Downsample to fMRI TR-rate

**Problem:** Word embeddings are at ~3 words/sec; fMRI scans every 2 s (TR=2).  
We need one embedding vector per TR, not one per word.

**Method:** Lanczos (sinc-based) interpolation via `lanczosinterp2D`.  
For each TR timepoint $t_k$, compute a weighted average of nearby word vectors:
$$x_{t_k} = \sum_i L(\tau_i - t_k) \cdot e_i$$
where $L$ is the Lanczos kernel and $\tau_i$ is the timestamp of word $i$.  
This is equivalent to low-pass filtering then resampling — it avoids aliasing.

Output: `{story: np.ndarray of shape (n_TRs, embed_dim)}`

In [41]:
# downsample_word_vectors uses wordseqs[story].data_times and .tr_times
downsampled = downsample_word_vectors(ALL_STORIES, word_vectors, wordseqs)

for story in ALL_STORIES:
    print(f'  {story}: {word_vectors[story].shape} words -> {downsampled[story].shape} TRs')

  mybackseatviewofagreatromance: (1794, 300) words -> (399, 300) TRs
  becomingindian: (2619, 300) words -> (403, 300) TRs
  cocoonoflove: (1984, 300) words -> (444, 300) TRs
  goldiethegoldfish: (1680, 300) words -> (332, 300) TRs
  theclosetthatateeverything: (1928, 300) words -> (329, 300) TRs
  quietfire: (1905, 300) words -> (470, 300) TRs
  lifeanddeathontheoregontrail: (2389, 300) words -> (376, 300) TRs
  thepostmanalwayscalls: (2220, 300) words -> (469, 300) TRs
  findingmyownrescuer: (1498, 300) words -> (387, 300) TRs
  sloth: (2403, 300) words -> (452, 300) TRs
  legacy: (1893, 300) words -> (415, 300) TRs
  treasureisland: (1763, 300) words -> (409, 300) TRs
  learninghumanityfromdogs: (1484, 300) words -> (328, 300) TRs
  naked: (3218, 300) words -> (437, 300) TRs
  hangtime: (1927, 300) words -> (339, 300) TRs
  lawsthatchokecreativity: (2084, 300) words -> (449, 300) TRs
  jugglingandjesus: (887, 300) words -> (208, 300) TRs
  whyimustspeakoutaboutclimatechange: (2336, 

---
## Stage 3 — Trim Edges + Add Hemodynamic Delays

### Trim
Remove boundary timepoints that are unreliable:
- **First 5 TRs:** scanner warmup; subject hasn't settled into listening
- **Last 10 TRs:** BOLD response lags the audio; last words not fully captured

### Delay (`make_delayed`)
The BOLD signal peaks ~4–6 s after a stimulus. `make_delayed` creates 4 shifted copies of
the embedding matrix (delays 1–4 TRs) stacked horizontally.  
This lets ridge regression learn the best delay profile for each voxel.

$$X_{\text{delayed}} = [X_{t-1} \;|\; X_{t-2} \;|\; X_{t-3} \;|\; X_{t-4}]$$

> **Note:** For delay `d`, the first `d` rows of that copy are **zero-padded** (no data
> exists before the start of the story). So the final matrix has a block of zeros in the
> top-left corner — delay-1 has 1 zero row, delay-2 has 2, …, delay-4 has 4.

Output shape: `(T', embed_dim × 4)` — e.g. `(T', 1200)` for GloVe 300-d

In [42]:
start_idx = TRIM_START
end_idx   = TRIM_END

# Story embeddings are subject-independent -- process once.
# Trim based on the story own TR count (from wordseq), not any subject fMRI length.
processed = {}
for story in ALL_STORIES:
    arr = downsampled[story]                         # (n_trs_seq, 300)
    arr = arr[start_idx : len(arr) - end_idx]        # (T_story, 300)
    arr = make_delayed(arr, DELAYS)                  # (T_story, 1200)
    processed[story] = arr
    print(f"  {story}: embedding {arr.shape}")

print()

# Per-subject: stack stories and save.
# fMRI (loaded separately via load_fmri) is trimmed the same way and should align.
for subject in SUBJECTS:
    sid = SUBJECT_IDS[subject]

    X_train = np.vstack([processed[s] for s in TRAIN_STORIES])
    X_test  = np.vstack([processed[s] for s in TEST_STORIES])

    train_path = os.path.join(EMBED_DIR, f"{sid}_train_glove_embeddings.npy")
    test_path  = os.path.join(EMBED_DIR, f"{sid}_test_glove_embeddings.npy")
    np.save(train_path, X_train)
    np.save(test_path,  X_test)

    print(f"Saved [{subject}]  X_train: {X_train.shape}  ->  {train_path}")
    print(f"Saved [{subject}]  X_test:  {X_test.shape}  ->  {test_path}")
    print()

  mybackseatviewofagreatromance: embedding (384, 1200)
  becomingindian: embedding (388, 1200)
  cocoonoflove: embedding (429, 1200)
  goldiethegoldfish: embedding (317, 1200)
  theclosetthatateeverything: embedding (314, 1200)
  quietfire: embedding (455, 1200)
  lifeanddeathontheoregontrail: embedding (361, 1200)
  thepostmanalwayscalls: embedding (454, 1200)
  findingmyownrescuer: embedding (372, 1200)
  sloth: embedding (437, 1200)
  legacy: embedding (400, 1200)
  treasureisland: embedding (394, 1200)
  learninghumanityfromdogs: embedding (313, 1200)
  naked: embedding (422, 1200)
  hangtime: embedding (324, 1200)
  lawsthatchokecreativity: embedding (434, 1200)
  jugglingandjesus: embedding (193, 1200)
  whyimustspeakoutaboutclimatechange: embedding (511, 1200)
  thetiniestbouquet: embedding (171, 1200)
  thumbsup: embedding (414, 1200)
  breakingupintheageofgoogle: embedding (521, 1200)
  gpsformylostidentity: embedding (326, 1200)
  escapingfromadirediagnosis: embedding (343, 1

In [43]:
# Verify zero-padding in make_delayed using a small toy example
toy = np.ones((6, 2), dtype=float)   # 6 timepoints, 2 features
toy_delayed = make_delayed(toy, DELAYS)

print('Toy input (6 x 2, all ones):')
print(toy)
print(f'\nAfter make_delayed with delays={DELAYS}  -> shape {toy_delayed.shape}')
print('(columns grouped as: [delay1 | delay2 | delay3 | delay4])')
print()
print(toy_delayed)
print()

# Count leading zero rows per delay block
dim = toy.shape[1]
for i, d in enumerate(DELAYS):
    block = toy_delayed[:, i*dim:(i+1)*dim]
    zero_rows = int(np.sum(~block.any(axis=1)[:d]))
    print(f'  delay={d}: first {d} row(s) zero-padded  ->  {zero_rows} zero row(s) confirmed')

Toy input (6 x 2, all ones):
[[1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]
 [1. 1.]]

After make_delayed with delays=[1, 2, 3, 4]  -> shape (6, 8)
(columns grouped as: [delay1 | delay2 | delay3 | delay4])

[[0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 0. 0. 0. 0. 0. 0.]
 [1. 1. 1. 1. 0. 0. 0. 0.]
 [1. 1. 1. 1. 1. 1. 0. 0.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]]

  delay=1: first 1 row(s) zero-padded  ->  1 zero row(s) confirmed
  delay=2: first 2 row(s) zero-padded  ->  2 zero row(s) confirmed
  delay=3: first 3 row(s) zero-padded  ->  3 zero row(s) confirmed
  delay=4: first 4 row(s) zero-padded  ->  4 zero row(s) confirmed


---
## Verification

Check that saved embeddings load correctly and match expected shapes.

In [44]:
for subject in SUBJECTS:
    sid = SUBJECT_IDS[subject]
    for split_name in ['train', 'test']:
        path = os.path.join(EMBED_DIR, f'{sid}_{split_name}_glove_embeddings.npy')
        arr  = np.load(path)
        expected_cols = GLOVE_DIM * len(DELAYS)   # 300 * 4 = 1200
        assert arr.shape[1] == expected_cols, f'Expected {expected_cols} cols, got {arr.shape[1]}'
        print(f'  {os.path.basename(path)}: {arr.shape}  OK')

  s2_train_glove_embeddings.npy: (27678, 1200)  OK
  s2_test_glove_embeddings.npy: (7108, 1200)  OK
  s3_train_glove_embeddings.npy: (27678, 1200)  OK
  s3_test_glove_embeddings.npy: (7108, 1200)  OK


---
## Summary

```
raw transcript
    │
    │  Stage 1 — GloVe lookup
    │  {story: (n_words, 300)}
    │
    │  Stage 2 — Lanczos downsample
    │  {story: (n_TRs, 300)}
    │
    │  Stage 3a — crop + trim
    │  {story: (T', 300)}
    │
    │  Stage 3b — make_delayed (×4)
    │  {story: (T', 1200)}
    │
    ▼
X_train / X_test .npy  →  ridge regression
```

| Stage | Shape | Notes |
|---|---|---|
| Word embedding | `(n_words, 300)` | GloVe lookup; OOV → 0 |
| Downsample | `(n_TRs, 300)` | Lanczos interpolation |
| Trim | `(T', 300)` | Remove first 2 and last 5 TRs |
| Delay | `(T', 1200)` | 4 lagged copies stacked |
| Stacked | `(ΣT', 1200)` | All stories concatenated |